# 🌦️ Notebook 05: Model Building — Advanced Track

## Overview
This notebook covers advanced forecasting models:
1. Prophet model (Facebook's decomposable forecasting)
2. XGBoost with lag features (tree-based)
3. Ensemble Stacking (combining all models)
4. Model comparison table

**Models:** Prophet, XGBoost, Ensemble  
**Metrics:** MAE, RMSE, R²  

---

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported")

In [ ]:
# Load data
df = pd.read_csv("../data/weather_cleaned.csv", parse_dates=["last_updated"])

city = "London"
city_data = df[df["location_name"] == city].copy()

# Prepare time-series
city_ts = (
    city_data.set_index("last_updated")["temperature_celsius"]
    .resample("D").mean().interpolate()
)

train_size = int(len(city_ts) * 0.8)
train_ts = city_ts.iloc[:train_size]
test_ts = city_ts.iloc[train_size:]

print(f"✅ Data loaded: {city}")
print(f"   Train: {len(train_ts)}, Test: {len(test_ts)}")

## Model 1: Prophet

In [ ]:
# Prepare Prophet DataFrame
prophet_df = city_ts.reset_index().rename(
    columns={"last_updated": "ds", "temperature_celsius": "y"}
)
train_prophet = prophet_df.iloc[:train_size]
test_prophet = prophet_df.iloc[train_size:]

# Fit Prophet
model_prophet = Prophet(
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

with open('/dev/null', 'w') as f:
    import sys
    old_stdout = sys.stdout
    sys.stdout = f
    model_prophet.fit(train_prophet)
    sys.stdout = old_stdout

# Forecast
future = model_prophet.make_future_dataframe(periods=len(test_prophet))
forecast_prophet = model_prophet.predict(future)

# Evaluate
pred_prophet = forecast_prophet.tail(len(test_prophet))["yhat"].values
mae_p = mean_absolute_error(test_prophet["y"], pred_prophet)
rmse_p = np.sqrt(mean_squared_error(test_prophet["y"], pred_prophet))
r2_p = r2_score(test_prophet["y"], pred_prophet)

print(f"✅ Prophet Model:")
print(f"   MAE: {mae_p:.3f}°C")
print(f"   RMSE: {rmse_p:.3f}°C")
print(f"   R²: {r2_p:.3f}")

## Model 2: XGBoost with Lag Features

In [ ]:
# Create lag features
def create_lag_features(series, lags=[1, 2, 3, 7, 14, 30]):
    df_lag = pd.DataFrame({"y": series})
    for lag in lags:
        df_lag[f"lag_{lag}"] = series.shift(lag)
    df_lag["rolling_mean_7"] = series.shift(1).rolling(7).mean()
    df_lag["rolling_std_7"] = series.shift(1).rolling(7).std()
    df_lag["rolling_mean_30"] = series.shift(1).rolling(30).mean()
    df_lag["month"] = series.index.month
    df_lag["day_of_week"] = series.index.dayofweek
    df_lag["day_of_year"] = series.index.dayofyear
    return df_lag.dropna()

lag_df = create_lag_features(city_ts)
X_lag = lag_df.drop("y", axis=1)
y_lag = lag_df["y"]

train_cutoff = int(len(lag_df) * 0.8)
X_train_xgb, X_test_xgb = X_lag.iloc[:train_cutoff], X_lag.iloc[train_cutoff:]
y_train_xgb, y_test_xgb = y_lag.iloc[:train_cutoff], y_lag.iloc[train_cutoff:]

# Fit XGBoost
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric="rmse",
    verbosity=0
)

xgb_model.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_test_xgb, y_test_xgb)],
    verbose=False
)

# Evaluate
pred_xgb = xgb_model.predict(X_test_xgb)
mae_xgb = mean_absolute_error(y_test_xgb, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, pred_xgb))
r2_xgb = r2_score(y_test_xgb, pred_xgb)

print(f"✅ XGBoost Model:")
print(f"   MAE: {mae_xgb:.3f}°C")
print(f"   RMSE: {rmse_xgb:.3f}°C")
print(f"   R²: {r2_xgb:.3f}")

## Feature Importance (XGBoost)

In [ ]:
# Get feature importance
importance = xgb_model.feature_importances_
feature_names = X_train_xgb.columns

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False).head(15)

# Plot
fig = px.bar(
    importance_df,
    x='importance',
    y='feature',
    orientation='h',
    title='XGBoost Feature Importance (Top 15)',
    template='plotly_dark'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.write_html("../reports/figures/13_xgboost_importance.html")
fig.show()

print(f"✅ Feature importance visualization saved")

## Model 3: Ensemble (Stacking)

## Model Comparison

In [ ]:
# Compile results
results = {
    "Model": ["Prophet", "XGBoost", "Ensemble"],
    "MAE": [mae_p, mae_xgb, mae_ens],
    "RMSE": [rmse_p, rmse_xgb, rmse_ens],
    "R2": [r2_p, r2_xgb, r2_ens]
}
results_df = pd.DataFrame(results)

print(f"\n🏆 Model Comparison:")
print(results_df.round(3).to_string(index=False))

# Save results
results_df.to_csv("../reports/advanced_model_results.csv", index=False)
print(f"\n✅ Results saved to reports/advanced_model_results.csv")

## Forecast Visualization

## Summary